# 94-867/19-867/12-768 
# From Data to Action: Designing Human-Centered, Impact-Driven Decision Systems

Student Name: `<FILL IN>`

Andrew ID: `<FILL IN>`

## Read This Carefully Before Start

Make sure write your name and andrew id above.

Make sure you FILL IN any place that instructs `YOUR CODE HERE`, `<FILL IN>`, or `YOUR ANSWER HERE`.

Questions entitled with a ✰ symbol means you have to include answers in your written submission. For your convenience, numbers of those questions are consistent with the handout PDF.

Before you turn these assignments in, make sure everything runs as expected. First, **Restart** the kernel and then **Run All** cells.

---


# Assignment 2-3: Decision‑Dependent Predict + Optimize — Surge Pricing with `gurobi-ML`

**Problem:** Ride-hailing marketplaces like Uber and Lyft use pricing as a mechanism, often called surge pricing, to balance the supply of available drivers with the demand for rides, with the goal to maximize the total number of order matches. 

During periods of high demand, a surge multiplier is applied to the base fare. A higher multiplier (e.g., 5) significantly affects rider-driver matching, while a very low multiplier (e.g., 1.01) provides little additional incentive for drivers and the platform. Therefore, it is important to determine the optimal surge price multiplier.

In this assignment, we assume that the surge multiplier ranges from a minimum of 1 (the base fare) to a maximum of 3.

**Goal:** Choose a surge multiplier `s ∈ [1, 3]` to maximize *successful matches* where matches = `min(supply, demand)`.

---
## Learning outcomes
- Use `gurobi-ML` to embed tree-based regressors inside a Gurobi model.
- Encode `matches = min(supply, demand)` and maximize it.
- Compare against a baseline (no surge) and visualize the results.



## 0) Requirements & Setup

- Python 3.9+
- Packages: `gurobipy`, `gurobi-ml`, `scikit-learn`, `numpy`, `pandas`, `matplotlib`

> **Install (example):**
> ```bash
> pip install gurobipy gurobi-ml scikit-learn numpy pandas matplotlib
> ```


In [ ]:
import gurobipy as gp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gurobi_ml import add_predictor_constr
from gurobipy import GRB
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted
from tqdm import tqdm

np.random.seed(42)  # ! DO NOT MODIFY THIS LINE


## 1) Read data

Notation and Corresponding Columns
- $t$: `time_of_day ∈ {0,...,23}`
- $d$: `day_of_week ∈ {0,...,6}`
- $w$: `weather_score ∈ [-1, +1]`
- $s$: `surge_multiplier ∈ [1, 3]`
- $y_S$: **supply** of available drivers 
- $y_D$: **demand** for rides

Ground-truth drivers **increase** with surge; passengers **decrease** with surge; both have time/weather patterns and noise.


In [ ]:
df = pd.read_csv('Data/surge_pricing_train.csv')
df


## 2) Train ML predictors

While supply and demand change with uncertainty, we can predict those two using available features, i.e., $t, d, w, s$. In this case, 2 prediction models use the same input. 

We now split the data by a train-test ratio of 3:1. For training data, there should be 3 dataset:
- `X_train`: Input of prediction model
- `ys_train`: Ground truth of supply
- `yd_train`: Ground truth of demand 

And the test data follows the same.

Note that in real-world practice, you may need another validation set to validate your model performance before testing.

In [ ]:
# ! DO NOT MODIFY THIS CELL

input_features = ['time_of_day', 'day_of_week', 'weather_score', 'surge_multiplier']
X_train, X_test, ys_train, ys_test = train_test_split(df[input_features], df['supply'],
                                                      test_size=0.25, random_state=1)
_, _, yd_train, yd_test = train_test_split(df[input_features], df['demand'],
                                           test_size=0.25, random_state=1)

We use `RandomForestRegressor` (tree ensembles are supported by `gurobi-ML`).  
We fit **two** models, both taking inputs `[time_of_day(t), day_of_week(d), weather_score(w), surge_multiplier(s)]`:
- `supply_model:` $y_{S} = f_{\theta_{1}}(t,d,w,s)$
- `demand_model:` $y_{D} = f_{\theta_{2}}(t,d,w,s)$

In [ ]:
# ! DO NOT MODIFY THE MODEL PARAMS
model_params = {
    'n_estimators': 60,
    'max_depth': 6,
    'random_state': 42,
}
supply_model = RandomForestRegressor(**model_params)
demand_model = RandomForestRegressor(**model_params)


# YOUR ANSWER HERE
# TODO Fit both models and make predictions on the test set
ys_pred = <FILL IN>
yd_pred = <FILL IN>
# YOUR ANSWER ENDS


check_is_fitted(supply_model)
check_is_fitted(demand_model)
print("Supply model R^2 (test):", round(r2_score(ys_test, ys_pred), 4))
print("Demand model R^2 (test):", round(r2_score(yd_test, yd_pred), 4))

### Export test results

In [ ]:
# ! DO NOT MODIFY THIS CELL OUTPUTTING ONE CSV DELIVERABLE

(pd.DataFrame({'supply_pred': ys_pred, 'demand_pred': yd_pred})
 .round(4).to_csv('prediction_test.csv', index=False))


## 3) Formulation (Decision‑Dependent Prediction)

Given `time_of_day`, `day_of_week`, `weather_score`, we want to decide the `surge_multiplier` for this time based on predicted `supply` and `demand`

Let decision variable be the surge multiplier `s ∈ [1, 3]`.  
We embed both predictors using `gurobi-ML` so their inputs include `s`.

We define `matches ≤ supply_pred` and `matches ≤ demand_pred`; maximizing `matches` forces `matches = min(supply_pred, demand_pred)`.

### Parameter:
- `number of matches:` $m$
- `amount of rider demands:` $y_{D}$
- `amount of driver supplys:` $y_{S}$

### Decision variables:
- `Surge price multiplier`: $s$

### Objective function and constraints:
- `Objective: maximize the total number of matches:` $\max m$
- `Constraints: total number of matches = min {# rider demand, # driver supply}:` $m \leq y_{D}$, $m \leq y_{S}$
- `Constraints: ML prediction model --- amount of rider demand:` $y_{D} = f_{\theta_{2}}(t,d,w,s)$
- `Constraints: ML prediction model --- amount of driver supply:` $y_{S} = f_{\theta_{1}}(t,d,w,s)$

In [ ]:
def optimize_surge_price(data: pd.Series, supply_model, demand_model) -> dict:
    """
    Given time_of_day, day_of_week, weather_score, optimize the surge multiplier

    Args:
        data (pd.Series): ["time_of_day", "day_of_week", "weather_score"]
        supply_model: Fitted random forest model for supply prediction
        demand_model: Fitted random forest model for demand prediction

    Returns:
        dict: Dictionary with optimized supply, demand, matches, and surge multiplier decisions
    """
    m = gp.Model("surge_predict_opt")
    m.Params.OutputFlag = 0

    # Decision variable: surge multiplier
    s = m.addVar(lb=1, ub=3, name="surge_multiplier")
    # Variables for input features in data
    t = m.addVar(lb=data["time_of_day"], ub=data["time_of_day"], name="time_of_day")
    d = m.addVar(lb=data["day_of_week"], ub=data["day_of_week"], name="day_of_week")
    w = m.addVar(lb=data["weather_score"], ub=data["weather_score"], name="weather_score")
    # Variables for predicted supply and demand
    yS = m.addVar(lb=0.0, name="supply_pred")
    yD = m.addVar(lb=0.0, name="demand_pred")
    # Variable for matched rides for objective function
    matches = m.addVar(lb=0.0, name="matches")

    # YOUR ANSWER HERE
    # TODO add constraint for driver supply from ML prediction model

    # TODO add constraint for rider demands from ML prediction model

    # TODO add constraint that total number of matches = min {# rider demand, # driver supply}

    # TODO set objective maximizing the total number of matches

    # YOUR ANSWER ENDS

    m.optimize()
    return dict(
        supply_opt=yS.X,
        demand_opt=yD.X,
        surge_multiplier_opt=s.X,
        matches_opt=matches.X,
    )


# We can test the optimize_surge_price function with the first row training data
sample = df.loc[0][["time_of_day", "day_of_week", "weather_score"]]
optimize_surge_price(sample, supply_model, demand_model)


## 4) Optimization with 10 scenarios

With the optimization function, let's try with 10 different times.

For these 10 data points, we only have known time_of_day, day_of_week, and weather_score. We want information about predicted supply and demand, optimized surge multiplier and matches .

In [ ]:
opti_df = pd.read_csv('Data/surge_pricing_opti.csv')
opti_df

In [ ]:
result = []
for i in tqdm(opti_df.index):
    decisions = optimize_surge_price(opti_df.loc[i], supply_model, demand_model)
    result.append(decisions)
decision_df = pd.concat([opti_df, pd.DataFrame(result)], axis=1)
decision_df

### Export decision results

In [ ]:
# ! DO NOT MODIFY THIS CELL OUTPUTTING ONE CSV DELIVERABLE

decision_df.round(4).to_csv('optimization_decisions.csv', index=False)


## 5) Post-Analysis

Run the remaining cells and answer the last written question in the handout

### Match Improvement

We can consider the situation of surge multiplier of 1 as the baseline where we can get the corresponding supply and demand prediction. In this case, the match would be the min between 2 predictions.

In [ ]:
decision_df['surge_multiplier'] = 1
decision_df['supply_base'] = supply_model.predict(decision_df[input_features])
decision_df['demand_base'] = demand_model.predict(decision_df[input_features])
decision_df['matches_base'] = np.minimum(decision_df['supply_base'], decision_df['demand_base'])
decision_df

Then, we can compare the optimization with the baseline.

In [ ]:
plt.figure(figsize=(9, 6))

decision_df['matches_base'].plot(marker='x', label='Baseline (surge multiplier=1)')
decision_df['matches_opt'].plot(marker='o', label='Optimization Model')

plt.xlabel('Scenario Index')
plt.ylabel('Number of Matches')
plt.grid(linestyle='--')
plt.legend()
plt.show()

### Dig into the first Scenario 

In [ ]:
scenario = decision_df.iloc[0]  # Get the first scenario
multipliers = np.linspace(1.0, 3.0, 201)  # Enumerate surge possible multipliers from 1 to 3
feats = np.column_stack([
    np.full_like(multipliers, scenario["time_of_day"]),
    np.full_like(multipliers, scenario["day_of_week"]),
    np.full_like(multipliers, scenario["weather_score"]),
    multipliers
])
supply_curve = supply_model.predict(feats)
demand_curve = demand_model.predict(feats)

plt.figure(figsize=(9, 6))
plt.title(f"Supply & Demand vs Surge Multiplier")
plt.plot(multipliers, supply_curve, label="supply_pred")
plt.plot(multipliers, demand_curve, label="demand_pred")
plt.axvline(scenario["surge_multiplier_opt"], linestyle="--", color="black",
            label="surge_multiplier_opt")
plt.xlabel("surge multiplier")
plt.ylabel("prediction")
plt.legend()
plt.show()